In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.rcParams.update({
    # Font family
    'font.family': 'serif',  # or 'sans-serif'
    'font.serif': ['Times New Roman', 'DejaVu Serif'],  # For serif
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],  # For sans-serif
    
    # Font sizes
    'font.size': 11,           # Base font size
    'axes.titlesize': 13,      # Title font size
    'axes.labelsize': 11,      # Axis label size
    'xtick.labelsize': 9,      # X-axis tick label size
    'ytick.labelsize': 9,      # Y-axis tick label size
    'legend.fontsize': 9,      # Legend font size
    'figure.titlesize': 14,    # Figure title size
    
    
    # Other useful settings
    'figure.dpi': 300,         # Default DPI for figures
    'savefig.dpi': 300,        # Save figures at 300 DPI
    'savefig.bbox': 'tight',   # Tight bounding box
    'savefig.pad_inches': 0.05, # Padding around figure
})

## Environment Setup & Dependencies

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
# !pip install scipy umap-learn tqdm pandas numpy openpyxl

import os, random, warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.manifold import TSNE
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


---
## Configuration Hub & Reproducibility


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional


@dataclass
class Config:
    # ── Data ────────────────────────────────────────────────────────────────
    data_dir        : str   = "Data/"
    en_file         : str   = "GoEmotions.xlsx"
    zh_file         : str   = "EmoTalkCN.xlsx"
    bn_file         : str   = "BEmoC.xlsx"
    text_col        : str   = "Text"
    label_col       : str   = "Emo_Lvl"

    # ── Emotion ontology ────────────────────────────────────────────────────
    emotions        : List[str] = field(default_factory=lambda: [
        "anger", "surprise", "joy", "sadness", "disgust", "fear"
    ])
    num_classes     : int   = 6

    # ── Encoder ─────────────────────────────────────────────────────────────
    encoder_name    : str   = "XLM-RoBERTa-base"
    max_seq_len     : int   = 128
    hidden_size     : int   = 768
    pool_strategy   : str   = "cls_mean"   # 'cls' | 'mean' | 'cls_mean'

    # ── Self-attention in classifier ────────────────────────────────────────
    attn_heads      : int   = 8
    attn_dropout    : float = 0.1

    # ── Dropout ─────────────────────────────────────────────────────────────
    dropout_rate    : float = 0.3

    # ── Training ────────────────────────────────────────────────────────────
    batch_size      : int   = 32
    epochs_zero     : int   = 8
    epochs_few      : int   = 15
    epochs_full     : int   = 25
    warmup_ratio    : float = 0.1
    max_grad_norm   : float = 1.0

    # ── Loss weights ────────────────────────────────────────────────────────
    lambda1         : float = 0.5   # prototype loss
    lambda2         : float = 0.3   # alignment loss
    lambda3         : float = 0.2   # consistency loss
    adv_weight      : float = 0.1   # adversarial loss

    # ── Few-shot ────────────────────────────────────────────────────────────
    few_shot_k      : int   = 10    # samples per class

    # ── Optimiser ───────────────────────────────────────────────────────────
    encoder_lr      : float = 2e-5
    head_lr         : float = 1e-4
    weight_decay    : float = 0.01
    eta_min         : float = 1e-6

    # ── Reproducibility ─────────────────────────────────────────────────────
    seed            : int   = 42

    # ── Output ──────────────────────────────────────────────────────────────
    output_dir      : str   = "outputs/"
    save_best       : bool  = True

CFG = Config()

def set_seed(seed: int = 42):
    """Global seed for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"]       = str(seed)

set_seed(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.data_dir).mkdir(parents=True, exist_ok=True)

LABEL2ID = {e: i for i, e in enumerate(CFG.emotions)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}

print("Configuration done")
print(f"  Encoder  : {CFG.encoder_name}")
print(f"  Device   : {DEVICE}")
print(f"  Emotions : {CFG.emotions}")
print(f"  Label map: {LABEL2ID}")


## Data Loading & Preprocessing

In [ ]:
def load_dataset(path: str, lang: str) -> pd.DataFrame:
    """Load CSV; fall back to synthetic if file not found."""
    if os.path.exists(path):
        df = pd.read_excel(path)
        print(f"  [{lang:>8}] Loaded from disk  →  {len(df):,} rows")
    else:
        print(f"  [{lang:>8}] ⚠ File not found — check data path!!!")
        # df = _make_synthetic_data(lang)
    # ── Validate schema ───────────────────────────────────────────────────
    assert CFG.text_col  in df.columns, f"Missing column: {CFG.text_col}"
    assert CFG.label_col in df.columns, f"Missing column: {CFG.label_col}"
    df = df[[CFG.text_col, CFG.label_col]].dropna()
    df[CFG.label_col] = df[CFG.label_col].str.lower().str.strip()
    df = df[df[CFG.label_col].isin(CFG.emotions)].reset_index(drop=True)
    df["label_id"] = df[CFG.label_col].map(LABEL2ID)
    df["lang"]     = lang.lower()
    return df

print("Loading corpora …")
df_en  = load_dataset(os.path.join(CFG.data_dir, CFG.en_file), "English")
df_zh  = load_dataset(os.path.join(CFG.data_dir, CFG.zh_file), "Chinese")
df_bn  = load_dataset(os.path.join(CFG.data_dir, CFG.bn_file), "Bangla")
df_all = pd.concat([df_en, df_zh, df_bn], ignore_index=True)

print(f"\nDataset summary:")
print(df_all.groupby(["lang", CFG.label_col]).size().unstack(fill_value=0))


In [ ]:
# ── EDA visualisation ─────────────────────────────────────────────────────
PALETTE = {
    "anger":"#E63946", "surprise":"#FFB703", "joy":"#2EC4B6",
    "sadness":"#457B9D", "disgust":"#6A4C93", "fear":"#8D99AE"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
langs_data = [("English", df_en), ("Chinese", df_zh), ("Bangla", df_bn)]
for ax, (lang, df) in zip(axes, langs_data):
    counts = df[CFG.label_col].value_counts().reindex(CFG.emotions, fill_value=0)
    bars   = ax.bar(counts.index, counts.values,
                    color=[PALETTE[e] for e in counts.index], edgecolor="white", linewidth=1.2)
    ax.set_title(f"{lang} Corpus", fontsize=13, fontweight="bold", pad=10)
    ax.set_xlabel("Emotion Class", fontsize=12)
    ax.set_ylabel("Sample Count",  fontsize=12)
    ax.tick_params(axis="x",  labelsize=12, rotation=30)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 5, str(v),
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylim(0, counts.max() * 1.15)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Cross-Lingual Emotion Class Distribution", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CFG.output_dir}eda_class_distribution.pdf", bbox_inches="tight", dpi=600)
plt.show()
print("EDA figure saved")


---
## Dataset Class & DataLoader Construction


In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class EmotionDataset(Dataset):
    """Tokenised emotion dataset for XLM-RoBERTa."""

    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_len: int = CFG.max_seq_len,
        augment: bool = False,
    ):
        self.texts     = df[CFG.text_col].tolist()
        self.labels    = df["label_id"].tolist()
        self.langs     = df["lang"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.augment   = augment

    def __len__(self):
        return len(self.texts)

    def _augment(self, text: str) -> str:
        """Light synonym-swap augmentation (token-level shuffle as proxy)."""
        tokens = text.split()
        if len(tokens) > 3 and random.random() < 0.15:
            i, j  = random.sample(range(len(tokens)), 2)
            tokens[i], tokens[j] = tokens[j], tokens[i]
        return " ".join(tokens)

    def __getitem__(self, idx):
        text  = self.texts[idx]
        if self.augment:
            text = self._augment(text)
        enc   = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "label"          : torch.tensor(self.labels[idx], dtype=torch.long),
            "lang"           : self.langs[idx],
        }


def make_balanced_sampler(dataset: EmotionDataset) -> WeightedRandomSampler:
    """Over/under-samples to create a balanced training distribution."""
    labels  = [dataset.labels[i] for i in range(len(dataset))]
    counts  = np.bincount(labels, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / counts
    sample_w= [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)


def build_splits(df_src: pd.DataFrame, df_tgt: pd.DataFrame, tokenizer,
                 scenario: str = "zero", few_k: int = CFG.few_shot_k):
    """
    Returns (train_loader, val_loader, test_loader) tuples.
    scenario ∈ {'zero', 'few', 'full'}
    """
    from sklearn.model_selection import train_test_split

    # ── Test set: Bangla only ─────────────────────────────────────────────
    tgt_train, tgt_test = train_test_split(
        df_tgt, test_size=0.2, stratify=df_tgt["label_id"], random_state=CFG.seed
    )

    if scenario == "zero":
        train_df = df_src
        val_df   = tgt_test.copy()         
    elif scenario == "few":
        few_samples = (
            tgt_train.groupby("label_id", group_keys=False)
                     .apply(lambda g: g.sample(min(few_k, len(g)), random_state=CFG.seed))
        )
        train_df = pd.concat([df_src, few_samples], ignore_index=True)
        val_df   = tgt_test.copy()
    else:  # full
        tgt_tr2, tgt_val = train_test_split(
            tgt_train, test_size=0.1, stratify=tgt_train["label_id"], random_state=CFG.seed
        )
        train_df = pd.concat([df_src, tgt_tr2], ignore_index=True)
        val_df   = tgt_val

    train_ds  = EmotionDataset(train_df, tokenizer, augment=(scenario != "zero"))
    val_ds    = EmotionDataset(val_df,   tokenizer)
    test_ds   = EmotionDataset(tgt_test, tokenizer)

    sampler   = make_balanced_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CFG.batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    print(f"  [{scenario:>5}] train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")
    return train_loader, val_loader, test_loader


# ── Tokenizer ─────────────────────────────────────────────────────────────
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(CFG.encoder_name)
print("Tokenizer ready")

# ── Merge source corpora ──────────────────────────────────────────────────
df_src = pd.concat([df_en, df_zh], ignore_index=True)

# ── Build loaders for all scenarios ───────────────────────────────────────
print("\nBuilding data loaders …")
loaders = {}
for scen, e_key in [("zero","epochs_zero"), ("few","epochs_few"), ("full","epochs_full")]:
    loaders[scen] = build_splits(df_src, df_bn, tokenizer, scenario=scen)
print("DataLoaders ready")


---
## Module 1 — Shared Encoder with FIX Pooling


In [ ]:
class FixPooling(nn.Module):

    def __init__(self, hidden_size: int = CFG.hidden_size):
        super().__init__()
        self.proj = nn.Linear(hidden_size * 2, hidden_size)   
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, last_hidden_state: torch.Tensor,
                attention_mask: torch.Tensor) -> torch.Tensor:
        # [CLS] token
        cls_rep  = last_hidden_state[:, 0, :]                

        # Masked mean pooling
        mask_exp = attention_mask.unsqueeze(-1).float()        
        sum_rep  = (last_hidden_state * mask_exp).sum(dim=1)   
        cnt      = mask_exp.sum(dim=1).clamp(min=1e-9)         
        mean_rep = sum_rep / cnt                               

        combined = torch.cat([cls_rep, mean_rep], dim=-1)     
        out      = self.norm(self.proj(combined))            
        return out                                            


class SharedEncoder(nn.Module):
    """
    Module 1: XLM-RoBERTa backbone + FIX Pooling.
    
    Supports layer-wise learning-rate decay (LLRD) for stable fine-tuning
    of the deep transformer stack on cross-lingual emotion data.
    """

    def __init__(self, model_name: str = CFG.encoder_name):
        super().__init__()
        self.encoder_model = AutoModel.from_pretrained(model_name)
        self.pooler        = FixPooling(CFG.hidden_size)
        self.output_dim    = CFG.hidden_size

    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.encoder_model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            output_hidden_states = False,
        )
        return self.pooler(outputs.last_hidden_state, attention_mask)

    def get_layerwise_params(self, base_lr: float = CFG.encoder_lr,
                             decay: float = 0.95):
        """
        Layer-wise learning-rate decay: deeper layers → lower LR.
        Prevents overwriting pre-trained multilingual knowledge.
        """
        num_layers = self.encoder_model.config.num_hidden_layers
        no_decay   = ["bias", "LayerNorm.weight"]
        groups     = []
        for layer_idx in range(num_layers, -1, -1):
            lr_i = base_lr * (decay ** (num_layers - layer_idx))
            if layer_idx == num_layers:
                named = [(n, p) for n, p in self.named_parameters()
                         if "encoder.layer" not in n]
            else:
                named = [(n, p) for n, p in self.named_parameters()
                         if f"encoder.layer.{layer_idx}." in n]
            if not named:
                continue
            groups += [
                {"params": [p for n, p in named if not any(nd in n for nd in no_decay)],
                 "lr": lr_i, "weight_decay": CFG.weight_decay},
                {"params": [p for n, p in named if     any(nd in n for nd in no_decay)],
                 "lr": lr_i, "weight_decay": 0.0},
            ]
        return groups

print("SharedEncoder (Module 1) defined")


---
## Module 2 — Emotion Classifier with Self-Attention



In [ ]:
class SelfAttentionClassifier(nn.Module):

    def __init__(
        self,
        input_dim  : int   = CFG.hidden_size,
        num_classes: int   = CFG.num_classes,
        num_heads  : int   = CFG.attn_heads,
        attn_drop  : float = CFG.attn_dropout,
        dropout    : float = CFG.dropout_rate,
    ):
        super().__init__()
        self.norm1     = nn.LayerNorm(input_dim)
        self.attn      = nn.MultiheadAttention(
            embed_dim   = input_dim,
            num_heads   = num_heads,
            dropout     = attn_drop,
            batch_first = True,
        )
        self.norm2     = nn.LayerNorm(input_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(input_dim, num_classes)

        # Learnable temperature for softmax calibration
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
      
        x_seq  = x.unsqueeze(1)
        normed = self.norm1(x_seq)

        # Self-attention over the single token 
        attn_out, attn_w = self.attn(normed, normed, normed)  
        attended = self.norm2(x_seq + attn_out).squeeze(1)    

        dropped  = self.dropout(attended)
        logits   = self.fc(dropped) / self.temperature.clamp(min=0.1)

        return {
            "logits"       : logits,                           
            "probs"        : F.softmax(logits, dim=-1),        
            "attn_weights" : attn_w.squeeze(-1),               
            "attended_repr": attended,                         #for prototype loss
        }

print("SelfAttentionClassifier (Module 2) defined")


---
## Module 3 — Emotion-Aware Cross-Lingual Alignment


In [ ]:
class GaussianMMDLoss(nn.Module):

    def __init__(self, kernel_bandwidths=(0.5, 1.0, 2.0, 4.0, 8.0)):
        super().__init__()
        self.bandwidths = kernel_bandwidths

    def _rbf_kernel(self, X: torch.Tensor, Y: torch.Tensor, bw: float) -> torch.Tensor:
        """Gaussian RBF kernel matrix."""
        XX = (X ** 2).sum(1, keepdim=True)
        YY = (Y ** 2).sum(1, keepdim=True)
        XY = X @ Y.T
        dist_sq = XX + YY.T - 2 * XY
        return torch.exp(-dist_sq / (2 * bw ** 2))

    def _mmd_sq(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        loss = torch.zeros(1, device=X.device)
        for bw in self.bandwidths:
            Kxx = self._rbf_kernel(X, X, bw)
            Kyy = self._rbf_kernel(Y, Y, bw)
            Kxy = self._rbf_kernel(X, Y, bw)
            loss = loss + Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()
        return loss / len(self.bandwidths)

    def forward(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        return self._mmd_sq(X, Y)


class EmotionAwareCrossLingualAlignment(nn.Module):

    def __init__(self, hidden_size: int = CFG.hidden_size, num_classes: int = CFG.num_classes):
        super().__init__()
        self.mmd    = GaussianMMDLoss()
        self.num_c  = num_classes
        self.align_proj = nn.Linear(hidden_size, hidden_size // 2, bias=False)

    def forward(
        self,
        src_feats : torch.Tensor,   
        tgt_feats : torch.Tensor,  
        src_probs : torch.Tensor,   # predicted probabilities
        tgt_probs : torch.Tensor,   
    ) -> torch.Tensor:

        src_proj = self.align_proj(src_feats)   
        tgt_proj = self.align_proj(tgt_feats)  

        total_loss = torch.zeros(1, device=src_feats.device)
        for c in range(self.num_c):
            # Soft-weighted centroids per class
            ws = src_probs[:, c].unsqueeze(1) + 1e-8   
            wt = tgt_probs[:, c].unsqueeze(1) + 1e-8   
            src_c = (src_proj * ws) / ws.sum()          
            tgt_c = (tgt_proj * wt) / wt.sum()         

            src_c_exp = src_c.expand_as(src_proj)
            tgt_c_exp = tgt_c.expand_as(tgt_proj)

            total_loss = total_loss + self.mmd(src_c_exp, tgt_c_exp)

        return total_loss / self.num_c

print("EmotionAwareCrossLingualAlignment (Module 3) defined")


---
## Module 4 — Prototype-Based Emotion Transfer



In [ ]:
class PrototypeEmotionTransfer(nn.Module):

    def __init__(
        self,
        hidden_size : int   = CFG.hidden_size,
        num_classes : int   = CFG.num_classes,
        momentum    : float = 0.999,    
        temperature : float = 0.07,    
        sep_margin  : float = 2.0,     
        alpha       : float = 0.5,
        beta        : float = 0.3,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.momentum    = momentum
        self.temperature = temperature
        self.sep_margin  = sep_margin
        self.alpha       = alpha
        self.beta        = beta

        # Learnable prototypes 
        self.prototypes  = nn.Parameter(
            F.normalize(torch.randn(num_classes, hidden_size), dim=-1)
        )
        # EMA shadow prototypes 
        self.register_buffer(
            "ema_prototypes",
            F.normalize(torch.randn(num_classes, hidden_size), dim=-1)
        )

    @torch.no_grad()
    def update_ema(self):
        """Momentum update of EMA prototypes (called after each step)."""
        m = self.momentum
        self.ema_prototypes.data = (
            m * self.ema_prototypes.data + (1 - m) * self.prototypes.data
        )

    def compactness_loss(self, features: torch.Tensor,
                         labels: torch.Tensor) -> torch.Tensor:
        """Pull each feature toward its class prototype."""
        protos = F.normalize(self.prototypes, dim=-1)
        feats  = F.normalize(features, dim=-1)
        loss   = torch.zeros(1, device=features.device)
        count  = 0
        for c in range(self.num_classes):
            mask = (labels == c)
            if mask.sum() == 0:
                continue
            dist = 1 - (feats[mask] * protos[c]).sum(dim=-1)   # cosine distance
            loss = loss + dist.mean()
            count += 1
        return loss / max(count, 1)

    def separation_loss(self) -> torch.Tensor:
        """Push prototypes apart: encourage inter-class distances > margin."""
        protos = F.normalize(self.prototypes, dim=-1)   
        sim    = protos @ protos.T                       # cosine similarity
        mask   = ~torch.eye(self.num_classes, dtype=torch.bool, device=protos.device)
        sim_off = sim[mask].view(self.num_classes, self.num_classes - 1)
        loss    = F.relu(sim_off + 1 - self.sep_margin / 2).mean()
        return loss

    def contrastive_loss(self, features: torch.Tensor,
                         labels: torch.Tensor) -> torch.Tensor:
        """
        NT-Xent contrastive loss over (feature, prototype) pairs.
        Positive: (feature_i, prototype_{label_i})
        Negative: all other prototypes
        """
        protos = F.normalize(self.prototypes, dim=-1)   
        feats  = F.normalize(features, dim=-1)        

        # Similarity to all prototypes
        sims   = feats @ protos.T / self.temperature
        loss   = F.cross_entropy(sims, labels)
        return loss

    def forward(
        self,
        features: torch.Tensor,     
        labels  : torch.Tensor,     
    ) -> Dict[str, torch.Tensor]:

        l_compact  = self.compactness_loss(features, labels)
        l_separate = self.separation_loss()
        l_contrast = self.contrastive_loss(features, labels)

        total = l_compact + self.alpha * l_separate + self.beta * l_contrast
        return {
            "proto_loss"     : total,
            "compact_loss"   : l_compact,
            "separate_loss"  : l_separate,
            "contrastive_loss": l_contrast,
        }

print("PrototypeEmotionTransfer (Module 4) defined")


---
## Module 5 — Language Adversarial Training


In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    """
    Gradient Reversal Layer (GRL).
    Forward pass : identity (x → x)
    Backward pass: gradient is reversed and scaled by -λ
    """

    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_: float) -> torch.Tensor:
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_, None


class GradientReversal(nn.Module):
    def __init__(self, lambda_: float = 1.0):
        super().__init__()
        self.lambda_ = lambda_

    def set_lambda(self, val: float):
        self.lambda_ = val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_)


def grl_schedule(epoch: int, total_epochs: int, gamma: float = 10.0) -> float:
    """
    Progressive GRL weight schedule (Ganin et al., 2016):
    λ(p) = 2/(1 + exp(-γ·p)) - 1,  p = epoch/total_epochs
    """
    p = epoch / max(total_epochs, 1)
    return (2.0 / (1.0 + np.exp(-gamma * p))) - 1.0


class LanguageAdversarialTrainer(nn.Module):
    """
    Module 5: Language Adversarial Discriminator with GRL.

    Given a feature vector the discriminator tries to predict which
    language (English=0, Chinese=1, Bangla=2) produced it.
    The GRL ensures the encoder learns language-invariant features.
    """

    LANG_MAP = {"english": 0, "chinese": 1, "bangla": 2}

    def __init__(self, hidden_size: int = CFG.hidden_size, num_langs: int = 3):
        super().__init__()
        self.grl  = GradientReversal(lambda_=1.0)
        self.disc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, num_langs),
        )

    def forward(
        self,
        features  : torch.Tensor,       
        lang_labels: torch.Tensor,       
    ) -> torch.Tensor:
        rev_feats = self.grl(features)
        logits    = self.disc(rev_feats)  # (B, 3)
        loss      = F.cross_entropy(logits, lang_labels)
        return loss

    @staticmethod
    def encode_langs(lang_list: list, device: torch.device) -> torch.Tensor:
        ids = [LanguageAdversarialTrainer.LANG_MAP.get(l, 0) for l in lang_list]
        return torch.tensor(ids, dtype=torch.long, device=device)

print("LanguageAdversarialTrainer (Module 5) defined")


---
## Full Model Assembly


In [ ]:
class CrossLingualEmotionModel(nn.Module):

    def __init__(self):
        super().__init__()
        # ── Module 1 
        self.encoder     = SharedEncoder(CFG.encoder_name)
        # ── Module 2 
        self.classifier  = SelfAttentionClassifier()
        # ── Module 3 
        self.aligner     = EmotionAwareCrossLingualAlignment()
        # ── Module 4 
        self.prototyper  = PrototypeEmotionTransfer()
        # ── Module 5
        self.adversary   = LanguageAdversarialTrainer()

        self.ce_loss     = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ── Forward ───────────────────────────────────────────────────────────
    def encode(self, input_ids, attention_mask) -> torch.Tensor:
        return self.encoder(input_ids, attention_mask)

    def classify(self, features: torch.Tensor) -> Dict[str, torch.Tensor]:
        return self.classifier(features)

    def compute_loss(
        self,
        src_batch : Dict,          # batched source data
        tgt_batch : Optional[Dict],# batched target data (None in zero-shot eval)
        epoch     : int = 0,
        total_epochs : int = 10,
    ) -> Dict[str, torch.Tensor]:
        
        device = next(self.parameters()).device

        src_feat = self.encode(src_batch["input_ids"], src_batch["attention_mask"])
        src_out  = self.classify(src_feat)
        src_logits, src_probs = src_out["logits"], src_out["probs"]
        src_repr = src_out["attended_repr"]

        cls_loss = self.ce_loss(src_logits, src_batch["labels"])
        proto_out  = self.prototyper(src_repr, src_batch["labels"])
        proto_loss = proto_out["proto_loss"]

        lam = grl_schedule(epoch, total_epochs)
        self.adversary.grl.set_lambda(lam)

        lang_ids = LanguageAdversarialTrainer.encode_langs(
            src_batch["langs"], device
        )
        adv_loss = self.adversary(src_repr, lang_ids)

        losses = {
            "cls_loss"   : cls_loss,
            "proto_loss" : proto_loss,
            "adv_loss"   : adv_loss,
            "align_loss" : torch.zeros(1, device=device),
            "cons_loss"  : torch.zeros(1, device=device),
        }

        if tgt_batch is not None:
            tgt_feat  = self.encode(tgt_batch["input_ids"], tgt_batch["attention_mask"])
            tgt_out   = self.classify(tgt_feat)
            tgt_probs = tgt_out["probs"]
            tgt_repr  = tgt_out["attended_repr"]

            align_loss = self.aligner(src_repr, tgt_repr, src_probs, tgt_probs)
            losses["align_loss"] = align_loss

            src_avg = src_probs.mean(dim=0, keepdim=True)   # (1, C)
            tgt_avg = tgt_probs.mean(dim=0, keepdim=True)   # (1, C)
            cons_loss = F.kl_div(
                tgt_avg.log().clamp(min=-1e4),
                src_avg,
                reduction="batchmean",
            )
            losses["cons_loss"] = cons_loss

            tgt_pseudo = tgt_probs.argmax(dim=-1)
            tgt_proto  = self.prototyper(tgt_repr, tgt_pseudo)
            losses["proto_loss"] = (proto_loss + tgt_proto["proto_loss"]) / 2

            tgt_lang_ids = LanguageAdversarialTrainer.encode_langs(
                tgt_batch["langs"], device
            )
            losses["adv_loss"] = (adv_loss + self.adversary(tgt_repr, tgt_lang_ids)) / 2

        # ── Total loss 
        total = (
            losses["cls_loss"]
            + CFG.lambda1 * losses["proto_loss"]
            + CFG.lambda2 * losses["align_loss"]
            + CFG.lambda3 * losses["cons_loss"]
            + CFG.adv_weight * losses["adv_loss"]
        )
        losses["total_loss"] = total
        return losses

print("CrossLingualEmotionModel defined")
print(f"  λ1 (proto)     = {CFG.lambda1}")
print(f"  λ2 (align)     = {CFG.lambda2}")
print(f"  λ3 (cons)      = {CFG.lambda3}")
print(f"  λ_adv          = {CFG.adv_weight}")


In [ ]:
model = CrossLingualEmotionModel().to(DEVICE)

def count_params(m: nn.Module, trainable_only=True):
    p = sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)
    return f"{p / 1e6:.2f}M"

print("Model parameter summary:")
print(f"  Total              : {count_params(model, trainable_only=False)}")
print(f"  Trainable          : {count_params(model)}")
print(f"  Encoder            : {count_params(model.encoder)}")
print(f"  Classifier         : {count_params(model.classifier)}")
print(f"  Aligner (M3)       : {count_params(model.aligner)}")
print(f"  Prototyper (M4)    : {count_params(model.prototyper)}")
print(f"  Adversary (M5)     : {count_params(model.adversary)}")



## Optimiser & Learning-Rate Scheduler



In [ ]:
def build_optimiser_and_scheduler(model: CrossLingualEmotionModel,
                                    total_steps: int):
    encoder_groups = model.encoder.get_layerwise_params(
        base_lr=CFG.encoder_lr, decay=0.95
    )

    head_groups = [
        {"params": model.classifier.parameters(),  "lr": CFG.head_lr},
        {"params": model.aligner.parameters(),      "lr": CFG.head_lr},
        {"params": model.prototyper.parameters(),   "lr": CFG.head_lr},
        {"params": model.adversary.parameters(),    "lr": CFG.head_lr * 0.5},
    ]

    all_groups = encoder_groups + head_groups
    optimizer  = AdamW(all_groups, eps=1e-8)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max   = total_steps,
        eta_min = CFG.eta_min,
    )
    return optimizer, scheduler

print("Optimiser builder defined")


## Training Engine


In [ ]:
from torch.cuda.amp import GradScaler, autocast

class EarlyStopping:
    """Stops training when validation metric does not improve for `patience` epochs."""
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = None
        self.counter    = 0
        self.should_stop = False

    def step(self, score: float) -> bool:
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


def run_epoch(
    model       : CrossLingualEmotionModel,
    src_loader  : DataLoader,
    tgt_loader  : Optional[DataLoader],   
    optimizer   : torch.optim.Optimizer,
    scheduler,
    scaler      : GradScaler,
    epoch       : int,
    total_epochs: int,
    training    : bool = True,
) -> Dict[str, float]:
    """Single epoch of training or evaluation."""
    model.train(training)
    totals  = {k: 0.0 for k in ["total","cls","proto","align","cons","adv"]}
    n_steps = 0

    tgt_iter = iter(tgt_loader) if tgt_loader else None

    for src_batch in src_loader:
        src_b = {
            "input_ids"   : src_batch["input_ids"].to(DEVICE),
            "attention_mask": src_batch["attention_mask"].to(DEVICE),
            "labels"      : src_batch["label"].to(DEVICE),
            "langs"       : src_batch["lang"],
        }

        tgt_b = None
        if tgt_iter:
            try:
                tgt_raw = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                tgt_raw  = next(tgt_iter)
            tgt_b = {
                "input_ids"     : tgt_raw["input_ids"].to(DEVICE),
                "attention_mask": tgt_raw["attention_mask"].to(DEVICE),
                "labels"        : tgt_raw["label"].to(DEVICE),
                "langs"         : tgt_raw["lang"],
            }

        if training:
            optimizer.zero_grad()
            with autocast(enabled=torch.cuda.is_available()):
                losses = model.compute_loss(src_b, tgt_b, epoch, total_epochs)
            scaler.scale(losses["total_loss"]).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            model.prototyper.update_ema()
        else:
            with torch.no_grad(), autocast(enabled=torch.cuda.is_available()):
                losses = model.compute_loss(src_b, tgt_b, epoch, total_epochs)

        for k, v in [
            ("total", losses["total_loss"]),
            ("cls",   losses["cls_loss"]),
            ("proto", losses["proto_loss"]),
            ("align", losses["align_loss"]),
            ("cons",  losses["cons_loss"]),
            ("adv",   losses["adv_loss"]),
        ]:
            totals[k] += v.item()
        n_steps += 1

    return {k: v / max(n_steps, 1) for k, v in totals.items()}


@torch.no_grad()
def evaluate(model: CrossLingualEmotionModel,
             loader: DataLoader) -> Dict[str, float]:
    """Compute accuracy, macro-F1, weighted-F1 on a DataLoader."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        feat = model.encode(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
        )
        logits = model.classify(feat)["logits"]
        preds  = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["label"].numpy())

    acc   = accuracy_score(all_labels, all_preds)
    mf1   = f1_score(all_labels, all_preds, average="macro",    zero_division=0)
    wf1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "macro_f1": mf1, "weighted_f1": wf1,
            "preds": all_preds, "labels": all_labels}


def train_model(
    model        : CrossLingualEmotionModel,
    train_loader : DataLoader,
    val_loader   : DataLoader,
    test_loader  : DataLoader,
    epochs       : int,
    scenario     : str,
    tgt_loader   : Optional[DataLoader] = None,
) -> Dict:
    """
    Full training pipeline for one scenario.
    Returns history dict with per-epoch metrics.
    """
    total_steps = len(train_loader) * epochs
    optimizer, scheduler = build_optimiser_and_scheduler(model, total_steps)
    scaler     = GradScaler(enabled=torch.cuda.is_available())
    stopper    = EarlyStopping(patience=3)
    best_mf1   = 0.0
    best_state = None
    history    = {"train": [], "val": [], "proto_pos": []}

    print(f"\n{'='*60}")
    print(f" Training Scenario: {scenario.upper()}")
    print(f" Epochs: {epochs} | Steps: {total_steps:,}")
    print(f"{'='*60}")

    for ep in range(1, epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(
            model, train_loader, tgt_loader,
            optimizer, scheduler, scaler,
            epoch=ep, total_epochs=epochs, training=True,
        )
        val_metrics = evaluate(model, val_loader)

        elapsed = time.time() - t0
        history["train"].append(train_loss)
        history["val"].append(val_metrics)

        with torch.no_grad():
            proto = F.normalize(model.prototyper.prototypes, dim=-1).cpu()
            spread = (proto.unsqueeze(0) - proto.unsqueeze(1)).norm(dim=-1).mean().item()
        history["proto_pos"].append(spread)

        print(
            f"Ep {ep:02d}/{epochs} | "
            f"L={train_loss['total']:.4f} "
            f"(cls={train_loss['cls']:.3f} "
            f"proto={train_loss['proto']:.3f} "
            f"align={train_loss['align']:.3f} "
            f"cons={train_loss['cons']:.3f}) | "
            f"Val MF1={val_metrics['macro_f1']:.4f} "
            f"Acc={val_metrics['accuracy']:.4f} | "
            f"{elapsed:.1f}s"
        )

        if val_metrics["macro_f1"] > best_mf1:
            best_mf1  = val_metrics["macro_f1"]
            best_state = deepcopy(model.state_dict())
            if CFG.save_best:
                torch.save(best_state, f"{CFG.output_dir}best_{scenario}.pt")

        if stopper.step(val_metrics["macro_f1"]):
            print(f"  Early stopping at epoch {ep}.")
            break

    if best_state:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader)
    history["test"] = test_metrics
    print(f"\n  ★ Test Results [{scenario.upper()}]:")
    print(f"    Accuracy    : {test_metrics['accuracy']:.4f}")
    print(f"    Macro F1    : {test_metrics['macro_f1']:.4f}  ← main metric")
    print(f"    Weighted F1 : {test_metrics['weighted_f1']:.4f}")
    return history

print("Training engine ready")


---
## Experiment Execution — Three Transfer Scenarios


In [ ]:
all_histories = {}
scenario_epochs = {
    "zero" : CFG.epochs_zero,
    "few"  : CFG.epochs_few,
    "full" : CFG.epochs_full,
}

In [ ]:
scenario = "zero"  # or "full"
epochs = CFG.epochs_zero  # or CFG.epochs_full

set_seed(CFG.seed)                                 
m = CrossLingualEmotionModel().to(DEVICE)
train_l, val_l, test_l = loaders[scenario]


tgt_for_train = val_l if scenario != "zero" else None

   

hist = train_model(
    model        = m,
    train_loader = train_l,
    val_loader   = val_l,
    test_loader  = test_l,
    epochs       = epochs,
    scenario     = scenario,
    tgt_loader   = tgt_for_train,
)
all_histories[scenario] = hist

all_histories[scenario]["model"] = m

In [ ]:
scenario = "full"  # or "zero",
epochs = CFG.epochs_full  # or CFG.epochs_zero,

set_seed(CFG.seed)                                   
m = CrossLingualEmotionModel().to(DEVICE)
train_l, val_l, test_l = loaders[scenario]


tgt_for_train = val_l if scenario != "zero" else None

   

hist = train_model(
    model        = m,
    train_loader = train_l,
    val_loader   = val_l,
    test_loader  = test_l,
    epochs       = epochs,
    scenario     = scenario,
    tgt_loader   = tgt_for_train,
)
all_histories[scenario] = hist
# Keep final models for visualisation
all_histories[scenario]["model"] = m

In [ ]:
type(all_histories)

df_h = pd.DataFrame.from_dict(all_histories, orient='index')
df_h.index.name = 'Scenario'
df_h.to_csv(f"{CFG.output_dir}all_histories.csv")
print(df_h)

---
## Results Summary Table



In [ ]:
# ── Per-scenario summary table ─────────────────────────────────────────────
rows = []
for scen, hist in all_histories.items():
    t = hist["test"]
    rows.append({
        "Scenario"   : scen.capitalize(),
        "Accuracy"   : f"{t['accuracy']:.4f}",
        "Macro F1"   : f"{t['macro_f1']:.4f}",
        "Weighted F1": f"{t['weighted_f1']:.4f}",
    })

df_res = pd.DataFrame(rows)
print("\n" + "="*52)
print(" Cross-Lingual Bangla Emotion Detection Results")
print("="*52)
print(df_res.to_string(index=False))

# ── Per-class F1 for the best scenario (full) ─────────────────────────────
best_hist = all_histories["full"]["test"]
preds, labels = best_hist["preds"], best_hist["labels"]
report = classification_report(
    labels, preds,
    target_names=CFG.emotions,
    digits=4,
    output_dict=True
)
df_report = pd.DataFrame(report).T.round(4)
print("\nPer-class classification report (Full Training):")
print(df_report.to_string())


---
## Statistical Significance Testing


In [ ]:
from sklearn.utils import resample

def bootstrap_f1(y_true, y_pred, n_boot=1000, seed=42):
    """
    Bootstrap confidence interval for Macro F1.
    Returns (mean, lower_95CI, upper_95CI, per-bootstrap scores).
    """
    rng    = np.random.default_rng(seed)
    scores = []
    n      = len(y_true)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    for _ in range(n_boot):
        idx  = rng.integers(0, n, size=n)
        s    = f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0)
        scores.append(s)
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5), scores


print("Statistical Significance Analysis")
print("="*55)

boot_results = {}
for scen, hist in all_histories.items():
    y_t = np.array(hist["test"]["labels"])
    y_p = np.array(hist["test"]["preds"])
    mean_f1, lo, hi, boot_scores = bootstrap_f1(y_t, y_p)
    boot_results[scen] = boot_scores
    print(f"  [{scen:>5}] Macro F1 = {mean_f1:.4f}  95% CI: [{lo:.4f}, {hi:.4f}]")

t_stat, p_val = stats.ttest_rel(boot_results["zero"], boot_results["full"])
print(f"\n  Paired t-test (zero vs full):")
print(f"    t-statistic = {t_stat:.4f}")
print(f"    p-value     = {p_val:.4e}")
print(f"    Significant = {'Yes ✓' if p_val < 0.05 else 'No ✗'} (α=0.05)")

w_stat, w_pval = stats.wilcoxon(boot_results["zero"], boot_results["full"])
print(f"\n  Wilcoxon signed-rank test (zero vs full):")
print(f"    statistic   = {w_stat:.4f}")
print(f"    p-value     = {w_pval:.4e}")
print(f"    Significant = {'Yes ✓' if w_pval < 0.05 else 'No ✗'} (α=0.05)")

fig, ax = plt.subplots(figsize=(9, 4))
colors  = {"zero": "#1526BA", "few": "#52D1E1", "full": "#2D920F"}
for scen, scores in boot_results.items():
    ax.hist(scores, bins=40, alpha=0.6, color=colors[scen],
            label=f"{scen.capitalize()}", edgecolor="white")
ax.set_xlabel("Bootstrap Macro F1", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Bootstrap Distribution of Macro F1 — All Scenarios", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{CFG.output_dir}bootstrap_f1_distribution.pdf", bbox_inches="tight", dpi=300)
plt.show()


---
## Confusion Matrix Analysis



In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax, normalise=True):
    """Plot a styled confusion matrix on a given axis."""
    cm = confusion_matrix(y_true, y_pred)
    if normalise:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    sns.heatmap(
        cm,
        annot      = True,
        fmt        = ".2f" if normalise else "d",
        cmap       = "Blues",
        xticklabels= CFG.emotions,
        yticklabels= CFG.emotions,
        ax         = ax,
        linewidths = 0.5,
        linecolor  = "white",
        annot_kws  = {"size": 12},
        vmin=0, vmax=1 if normalise else None,
    )
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.tick_params(axis="x",  labelsize=12, rotation=45)
    ax.tick_params(axis="y",  labelsize=12, rotation=0)


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = {"zero": "Zero-Shot Transfer", "few": "Few-Shot (k=10)", "full": "Full Training"}

for ax, (scen, title) in zip(axes, titles.items()):
    t = all_histories[scen]["test"]
    plot_confusion_matrix(t["labels"], t["preds"], title, ax)

# ── Highlight hard pairs ───────────────────────────────────────────────────
hard_pairs = [
    (CFG.emotions.index("fear"),    CFG.emotions.index("surprise")),
    (CFG.emotions.index("sadness"), CFG.emotions.index("disgust")),
]
for ax in axes:
    for (r, c) in hard_pairs:
        ax.add_patch(plt.Rectangle((c, r), 1, 1, fill=False,
                                    edgecolor="#F9F9F9", lw=2.5, zorder=5))
        ax.add_patch(plt.Rectangle((r, c), 1, 1, fill=False,
                                    edgecolor="#FBFBFB", lw=2.5, zorder=5))

from matplotlib.patches import Patch
legend_elem = [Patch(facecolor="none", edgecolor="#E63946", linewidth=2.5,
                     label="Hard pair (fear/surprise, sadness/disgust)")]
axes[0].legend(handles=legend_elem, loc="lower left", fontsize=11,
               bbox_to_anchor=(0, -0.3))

fig.suptitle("Confusion Matrices — Bangla Emotion Detection (Normalised)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CFG.output_dir}confusion_matrices.pdf", bbox_inches="tight", dpi=600)
plt.show()
print("Confusion matrices saved ✓")


---
## Hard Emotion Pair Analysis


In [ ]:
def hard_pair_analysis(y_true, y_pred, pair, label):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    a, b   = pair
    
    mask   = np.isin(y_true, [a, b])
    if mask.sum() == 0:
        return None
    yt, yp = y_true[mask], y_pred[mask]
    
    a_mask = (yt == a)
    b_mask = (yt == b)
    a_to_b = (yp[a_mask] == b).mean() if a_mask.sum() else 0.0
    b_to_a = (yp[b_mask] == a).mean() if b_mask.sum() else 0.0
    f1_a   = f1_score(yt, yp, labels=[a], average="micro", zero_division=0)
    f1_b   = f1_score(yt, yp, labels=[b], average="micro", zero_division=0)
    return {
        "pair"    : label,
        f"{CFG.emotions[a]}→{CFG.emotions[b]}_conf": f"{a_to_b:.3f}",
        f"{CFG.emotions[b]}→{CFG.emotions[a]}_conf": f"{b_to_a:.3f}",
        f"F1_{CFG.emotions[a]}": f"{f1_a:.4f}",
        f"F1_{CFG.emotions[b]}": f"{f1_b:.4f}",
    }

print("Hard Emotion Pair Analysis — Full Training Scenario")
print("="*60)
best_t = all_histories["full"]["test"]
for pair, name in [
    ((CFG.emotions.index("fear"), CFG.emotions.index("surprise")), "Fear ↔ Surprise"),
    ((CFG.emotions.index("sadness"), CFG.emotions.index("disgust")), "Sadness ↔ Disgust"),
]:
    r = hard_pair_analysis(best_t["labels"], best_t["preds"], pair, name)
    if r:
        for k, v in r.items():
            print(f"  {k:40s}: {v}")
        print()

# ── Bar chart of per-class F1 across scenarios ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(CFG.emotions))
w     = 0.25
scens = ["few", "zero", "full"]
c_map = {"zero":"#457B9D","few":"#FFB703","full":"#2EC4B6"}

for i, scen in enumerate(scens):
    t    = all_histories[scen]["test"]
    f1s  = [f1_score(t["labels"], t["preds"],
                     labels=[c], average="micro", zero_division=0)
            for c in range(CFG.num_classes)]
    bars = ax.bar(x + i*w - w, f1s, width=w, label=scen.capitalize(),
                  color=c_map[scen], edgecolor="white", linewidth=0.8)

# Highlight hard pairs
for idx in [CFG.emotions.index("fear"), CFG.emotions.index("surprise"),
            CFG.emotions.index("sadness"), CFG.emotions.index("disgust")]:
    ax.axvspan(idx - 0.45, idx + 0.45, alpha=0.07, color="#E63946", zorder=0)

ax.set_xticks(x); ax.set_xticklabels(CFG.emotions, fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("Per-Class F1 Scores Across Transfer Scenarios\n(shaded = hard emotion pairs)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.set_ylim(0, 1.05)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{CFG.output_dir}perclass_f1_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()


---
## t-SNE Embedding Visualisation


In [ ]:
@torch.no_grad()
def extract_features(model: CrossLingualEmotionModel,
                     df    : pd.DataFrame,
                     tokenizer,
                     n_samples: int = 300) -> tuple:
    """Extract pooled features and labels from a dataframe sample."""
    model.eval()
    df_s = df.sample(min(n_samples, len(df)), random_state=CFG.seed).reset_index(drop=True)
    ds   = EmotionDataset(df_s, tokenizer)
    dl   = DataLoader(ds, batch_size=64, shuffle=False)
    feats, lbls, langs = [], [], []
    for batch in dl:
        f = model.encode(batch["input_ids"].to(DEVICE),
                         batch["attention_mask"].to(DEVICE))
        feats.append(f.cpu().numpy())
        lbls.extend(batch["label"].numpy())
        langs.extend(batch["lang"])
    return np.vstack(feats), np.array(lbls), np.array(langs)


def plot_tsne(feats, labels_emo, labels_lang, title_suffix=""):
    """Side-by-side t-SNE plots: emotion view and language view."""
    print(f"  Running t-SNE … ({len(feats)} points)")
    tsne   = TSNE(n_components=2, perplexity=30, random_state=CFG.seed,
                  n_iter=1000, learning_rate="auto", init="pca")
    coords = tsne.fit_transform(feats)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ── (a) Emotion view ────────────────────────────────────────────────
    ax = axes[0]
    for i, emo in enumerate(CFG.emotions):
        mask = (labels_emo == i)
        shapes = ['o', 's', '^', 'D', 'v', 'p']  # Define shapes
        ax.scatter(coords[mask, 0], coords[mask, 1],
           c=PALETTE[emo], 
           marker=shapes[i],  # Add this line
           label=emo.capitalize(), s=50, alpha=0.75,
           edgecolors="black", linewidth=0.5)  # Add edgecolors
    ax.set_title(f"t-SNE: Coloured by Emotion{title_suffix}", fontsize=14, fontweight="bold")
    ax.legend(markerscale=2.2, fontsize=14, bbox_to_anchor=(1.02, 1), borderaxespad=0)
    ax.axis("off")

    # ── (b) Language view ───────────────────────────────────────────────
    ax = axes[1]
    lang_palette = {"english":"#E63946","chinese":"#2EC4B6","bangla":"#6A4C93"}
    for lng in ["english","chinese","bangla"]:
        mask = (labels_lang == lng)
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=lang_palette[lng], label=lng.capitalize(), s=25, alpha=0.75,
                   edgecolors="none")
    ax.set_title(f"t-SNE: Coloured by Language{title_suffix}", fontsize=13, fontweight="bold")
    ax.legend(markerscale=1.8, fontsize=9, bbox_to_anchor=(1.02, 1), borderaxespad=0)
    ax.axis("off")

    plt.suptitle("Cross-Lingual Emotion Embedding Space (XLM-RoBERTa + FIX Pooling)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    fname = f"{CFG.output_dir}tsne{title_suffix.replace(' ','_').replace('(','').replace(')','')}.pdf"
    plt.savefig(fname, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"  t-SNE saved: {fname} ✓")


# ── Extract & combine features for Full scenario model ─────────────────────
best_model = all_histories["full"]["model"]

print("Extracting features for t-SNE …")
f_en, l_en, ln_en = extract_features(best_model, df_en, tokenizer, n_samples=250)
f_zh, l_zh, ln_zh = extract_features(best_model, df_zh, tokenizer, n_samples=250)
f_bn, l_bn, ln_bn = extract_features(best_model, df_bn, tokenizer, n_samples=250)

all_feats = np.vstack([f_en, f_zh, f_bn])
all_emo   = np.concatenate([l_en, l_zh, l_bn])
all_lang  = np.concatenate([ln_en, ln_zh, ln_bn])

plot_tsne(all_feats, all_emo, all_lang, title_suffix=" (Full Training)")
